<a href="https://colab.research.google.com/github/Not-kh-lily-23/pulsar-conformal-triage/blob/main/data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import urllib.request
import zipfile
import pandas as pd
from sklearn.model_selection import train_test_split
base="pulsar-conformal-triage"
data=os.path.join(base,"data/raw")
processed=os.path.join(base,"data/processed")
src=os.path.join(base,"src")
os.makedirs(data,exist_ok=True)
os.makedirs(processed,exist_ok=True)
os.makedirs(src,exist_ok=True)
url="https://archive.ics.uci.edu/static/public/372/htru2.zip"
zip_path=os.path.join(data,"htru2.zip")
print("fetching HTRU2 dataset")
urllib.request.urlretrieve(url,zip_path)
with zipfile.ZipFile(zip_path,'r') as zip_ref:
    zip_ref.extractall(data)
columns=[
    "mean_profile","std_profile","kurtosis_profile","skewness_profile","mean_dmsnr","std_dmsnr","kurtosis_dmsnr","skewness_dmsnr","target"
]
csv_file=[os.path.join(data,f) for f in os.listdir(data) if f.endswith('.csv')][0]
df=pd.read_csv(csv_file,header=None,names=columns)
print(f"Total Records: {len(df)}")
print(f"Total Pulsars (Class 1): {df['target'].sum()} ({df['target'].mean()*100:.2f}%)\n")
x=df.drop(columns=["target"])
y=df["target"]
x_temp,x_tst,y_temp,y_tst=train_test_split(
    x,y,test_size=0.20,random_state=42,stratify=y
)
x_trn,x_calib,y_trn,y_calib=train_test_split(
    x_temp,y_temp,test_size=0.25,random_state=42,stratify=y_temp
)
x_trn.assign(target=y_trn).to_csv(f"{processed}/train.csv",index=False)
x_calib.assign(target=y_calib).to_csv(f"{processed}/calib.csv",index=False)
x_tst.assign(target=y_tst).to_csv(f"{processed}/test.csv",index=False)
splits=[("Train",y_trn),("Calib",y_calib),("Test",y_tst)]
for name, target in splits:
    pct = target.mean() * 100
    print(f"{name} Split: {len(target)} rows | Pulsar Ratio: {pct:.2f}%")
print("done")

fetching HTRU2 dataset
Total Records: 17898
Total Pulsars (Class 1): 1639 (9.16%)

Train Split: 10738 rows | Pulsar Ratio: 9.15%
Calib Split: 3580 rows | Pulsar Ratio: 9.16%
Test Split: 3580 rows | Pulsar Ratio: 9.16%
done
